In [1]:
# ============================================================
# APLIKASI PENJADWALAN KULIAH OTOMATIS
# GOOGLE COLAB + GRADIO + OR-TOOLS CP-SAT
# INPUT : 3 FILE EXCEL
# OUTPUT: 1 FILE EXCEL JADWAL BERWARNA
# ============================================================

# Jalankan di Google Colab. Instalasi otomatis:
!pip -q install -U ortools gradio pandas openpyxl

import os
import re
import math
import colorsys
import unicodedata
from collections import defaultdict

import pandas as pd
import gradio as gr
from ortools.sat.python import cp_model
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# ============================================================
# 1. KONFIGURASI WAKTU DAN OUTPUT
# ============================================================

HARI = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat"]
JAM_AWAL = 7
JAM_AKHIR = 17
SLOT = JAM_AKHIR - JAM_AWAL       # 10 slot, masing-masing 1 jam
OUTPUT = "/content/hasil_jadwal.xlsx"

# JP per kelas dikonversi menjadi menit (1 JP = 50 menit),
# kemudian dibulatkan ke atas menjadi slot 1 jam.
# Contoh: 8 JP = 400 menit = 6 jam 40 menit -> 7 slot.
# Proyek Informatika: 8 JP / 2 kelas = 4 JP per kelas
# = 200 menit -> 4 slot.
DURASI_BERDASARKAN_JP = False
MENIT_PER_JP = 50


# ============================================================
# 2. FUNGSI BANTU PEMBACAAN EXCEL
# ============================================================

def norm(x):
    """Normalisasi teks untuk pencocokan nama/header."""
    if x is None or pd.isna(x):
        return ""
    x = unicodedata.normalize("NFKD", str(x))
    x = "".join(c for c in x if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", x.lower().replace("\u2060", ""))


def txt(x):
    if x is None or pd.isna(x):
        return ""
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    return str(x).replace("\u2060", "").strip()


def angka(x, default=None):
    if x is None or pd.isna(x) or str(x).strip() == "":
        return default
    s = str(x).strip().replace(",", ".")
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group()) if m else default


def cari_header(df, kolom_wajib):
    target = {norm(x) for x in kolom_wajib}
    for i in range(len(df)):
        isi = {norm(x) for x in df.iloc[i].tolist() if not pd.isna(x)}
        if target.issubset(isi):
            return i
    return None


def header_mapping(row):
    """Membuat peta nama kolom -> indeks kolom."""
    hasil = {}
    for i, value in enumerate(row):
        k = norm(value)
        if k:
            hasil[k] = i
    return hasil


def ambil(row, mapping, nama, default=None):
    idx = mapping.get(norm(nama))
    if idx is None or idx >= len(row):
        return default
    return row[idx]


def label_kelas(i):
    """0=A, 1=B, ..., 26=AA."""
    hasil = ""
    i += 1
    while i:
        i, sisa = divmod(i - 1, 26)
        hasil = chr(65 + sisa) + hasil
    return hasil


# ============================================================
# 3. NORMALISASI NAMA DOSEN DAN ALIAS MATA KULIAH
# ============================================================

ALIAS_DOSEN = {
    norm("Agung"): "Agung Hernawan",
    norm("Eka"): "Eka Priyatma",
    norm("Bagas"): "Bagaskara",
    norm("Romo Kun"): "Kuntoro Adi",
    norm("Rm Kun"): "Kuntoro Adi",
    norm("Rm Hari"): "Hari Suparwito",
    norm("Rm. Hari"): "Hari Suparwito",
    norm("Romo Hari"): "Hari Suparwito",
    norm("Hari Supar"): "Hari Suparwito",
    norm("Pak Albert"): "Albert",
    norm("Romo Eko"): "Eko Hari",
}


def nama_dosen_baku(nama):
    nama = txt(nama)
    return ALIAS_DOSEN.get(norm(nama), nama)


COURSE_ALIASES = {
    norm("Statistik dan Probabilitas"): [
        norm("Pengantar Probabilitas")
    ],
    norm("Pendampingan Mahasiswa Baru 1 (FCH1)"): [
        norm("DPA 2026 (FCH 1)")
    ],
    norm("Pemrograman Analisa Data"): [
        norm("Pemrograman Analisis Data")
    ],
    norm("Generatif AI"): [
        norm("Generative AI")
    ],
}


# ============================================================
# 4. KATEGORI RUANG BERDASARKAN MATA KULIAH
# ============================================================

# Kategori mata kuliah:
# LAB              : Lab Basis Data atau LKD/lab komputer biasa.
# LAB_BASIS_DATA   : HANYA Lab Basis Data A-D.
# LAB_JARINGAN     : HANYA Lab Jaringan Komputer A-E.
# LAB_ANY          : Boleh menggunakan semua jenis laboratorium.
# KELAS            : HANYA ruang kuliah biasa (K.204, K.205, dst.).

DAFTAR_LAB_JARINGAN = {
    norm("Virtualisasi dan Cloud"),
    norm("Pemrograman Berbasis Web"),
    norm("Internet dan Aplikasinya"),
    norm("Protokol Internet"),
    norm("Protokol TCP/IP Suite"),
    norm("Jaringan Toleransi Tunda"),
    norm("Pengukuran dan Analisis Unjuk Kerja Jaringan"),
}

DAFTAR_LAB_BASIS_DATA = {
    norm("Sistem Tertanam"),
    norm("Pemerolehan Informasi"),
    norm("Administrasi Basis Data"),
}

DAFTAR_LAB_ANY = {
    norm("Pemrograman Perangkat Bergerak"),
    norm("Pemrograman Berbasis Bergerak"),
    norm("IoT"),
    norm("Internet of Things"),
}

DAFTAR_LAB_BIASA = {
    norm("Alpro"),
    norm("Algoritma dan Pemrograman"),
    norm("Algoritma dan Pemrograman Dasar"),
    norm("Proyek Informatika"),
    norm("Pemrograman Web"),
    norm("Pembelajaran Mesin"),
    norm("Pemrograman Analisis Data"),
    norm("Pemrograman Analisa Data"),
    norm("Komputer Pelihat"),
    norm("Komputer Penglihatan"),
    norm("Computer Vision"),
    norm("NLP"),
    norm("Pemrosesan Bahasa Alami"),
    norm("Pemrosesan Bahasa Alami (NLP)"),
    norm("Pemrosesan Citra"),
    norm("Generatif AI"),
    norm("Generative AI"),
    norm("Manajemen Data"),
    norm("Bahasa Query"),
}


def jenis_mata_kuliah(nama):
    n = norm(nama)

    # Prioritas aturan yang lebih spesifik harus lebih dahulu.
    if n in DAFTAR_LAB_JARINGAN:
        return "LAB_JARINGAN"

    if n in DAFTAR_LAB_BASIS_DATA:
        return "LAB_BASIS_DATA"

    if n in DAFTAR_LAB_ANY:
        return "LAB_ANY"

    if n in DAFTAR_LAB_BIASA:
        return "LAB"

    # Variasi penulisan nama mata kuliah.
    if "pemrosesanbahasaalami" in n or n.endswith("nlp"):
        return "LAB"

    if "komputerpelihat" in n or "komputerpenglihatan" in n:
        return "LAB"

    if "pemrogramananalisa" in n or "pemrogramananalisisdata" in n:
        return "LAB"

    if "generativeai" in n or "generatifai" in n:
        return "LAB"

    if n in {"iot", "internetofthings"}:
        return "LAB_ANY"

    if "sistemtertanam" in n or "pemerolehaninformasi" in n:
        return "LAB_BASIS_DATA"

    if "administrasibasisdata" in n:
        return "LAB_BASIS_DATA"

    if "pemrogramanperangkatbergerak" in n or "pemrogramanberbasisbergerak" in n:
        return "LAB_ANY"

    return "KELAS"


def jenis_ruangan(nama, jenis_excel=""):
    """Klasifikasi ruang berdasarkan kolom jenis dan nama ruang."""
    n = norm(nama)
    j = norm(jenis_excel)

    # Klasifikasi spesifik dari nama ruang.
    if "labbasisdata" in n or "laboratoriumbasisdata" in n:
        return "LAB_BASIS_DATA"

    if "jaringan" in n or "network" in n:
        return "LAB_JARINGAN"

    if "lab" in n or "laboratorium" in n or n.startswith("lkd"):
        return "LAB_BIASA"

    # Jika nama ruang tidak menunjukkan jenis, gunakan kolom jenis Excel.
    if j:
        if "basisdata" in j:
            return "LAB_BASIS_DATA"
        if "jaringan" in j or "network" in j:
            return "LAB_JARINGAN"
        if "lab" in j or "laboratorium" in j:
            return "LAB_BIASA"
        if "kelas" in j or "kuliah" in j or "teori" in j:
            return "KELAS"

    # Ruang kuliah biasa.
    return "KELAS"


def ruang_diizinkan(jenis_mk, ruang):
    """Hard filter kecocokan jenis mata kuliah dan jenis ruang."""
    jenis_ruang = ruang["jenis"]

    if jenis_mk == "LAB_JARINGAN":
        return jenis_ruang == "LAB_JARINGAN"

    if jenis_mk == "LAB_BASIS_DATA":
        return jenis_ruang == "LAB_BASIS_DATA"

    if jenis_mk == "LAB_ANY":
        return jenis_ruang in {
            "LAB_BASIS_DATA",
            "LAB_BIASA",
            "LAB_JARINGAN",
        }

    if jenis_mk == "LAB":
        return jenis_ruang in {
            "LAB_BASIS_DATA",
            "LAB_BIASA",
        }

    return jenis_ruang == "KELAS"


# ============================================================
# 5. IDENTIFIKASI TIGA FILE EXCEL
# ============================================================

def identifikasi_file(files):
    if not files:
        raise ValueError("Upload tiga file Excel terlebih dahulu.")

    if isinstance(files, str):
        files = [files]

    hasil = {}

    for file in files:
        xls = pd.ExcelFile(file)

        for sheet in xls.sheet_names:
            raw = pd.read_excel(
                file,
                sheet_name=sheet,
                header=None,
                dtype=object
            )

            if cari_header(
                raw,
                ["Nama Dosen", "Mata Kuliah", "Jumlah Kelas"]
            ) is not None:
                jenis = "dosen"

            elif cari_header(
                raw,
                ["Nama Mata Kuliah", "Rencana kelas", "Dosen Pengampu MK"]
            ) is not None:
                jenis = "kebutuhan"

            elif cari_header(raw, ["Ruang", "Kapasitas"]) is not None:
                jenis = "ruang"

            else:
                continue

            if jenis in hasil:
                raise ValueError(
                    f"Ditemukan lebih dari satu sheet/file untuk data {jenis}. "
                    "Pastikan hanya satu file untuk setiap jenis input."
                )

            hasil[jenis] = raw

    kurang = {"dosen", "kebutuhan", "ruang"} - set(hasil)

    if kurang:
        raise ValueError(
            "File belum lengkap. Harus ada tiga Excel:\n"
            "1. Mata Kuliah.xlsx\n"
            "2. kebutuhan kapasitas kelas tiap mata kuliah.xlsx\n"
            "3. kapasitas ruang.xlsx\n\n"
            f"Data yang belum ditemukan: {', '.join(sorted(kurang))}"
        )

    return hasil


# ============================================================
# 6. MEMBACA DATA DOSEN DAN JP DARI MATA KULIAH.XLSX
# ============================================================

def baca_database_dosen(raw):
    """
    File Mata Kuliah.xlsx berisi daftar dosen, mata kuliah, SKS, JP,
    dan jumlah kelas. Header dapat berulang untuk setiap dosen.
    Nama dosen yang kosong karena merge cell akan mengikuti nama
    dosen terakhir yang terbaca.
    """
    mapping = defaultdict(list)
    nama_aktif = ""

    for _, row in raw.iterrows():
        v = row.tolist()
        v += [None] * max(0, 6 - len(v))

        if norm(v[0]) in {"no", "nomor"} and norm(v[1]) == "namadosen":
            continue

        nama = txt(v[1])
        mata_kuliah = txt(v[2])

        if nama and norm(nama) not in {"jumlah", "total", "nan"}:
            nama_aktif = nama_dosen_baku(nama)

        if not nama_aktif or not mata_kuliah:
            continue

        if norm(mata_kuliah) in {"jumlah", "total"}:
            continue

        jp = angka(v[4], None)
        jumlah_kelas = angka(v[5], None)
        sks = angka(v[3], None)

        # Baris jabatan/struktural tidak dimasukkan karena jumlah kelas kosong.
        if jumlah_kelas is None or jumlah_kelas <= 0:
            continue

        mapping[norm(mata_kuliah)].append({
            "dosen": nama_aktif,
            "jp_total": jp,
            "sks": sks,
            "jumlah_kelas": int(jumlah_kelas),
        })

    return mapping


def cari_data_dosen_mk(nama_mk, database):
    key = norm(nama_mk)

    if key in database:
        return list(database[key])

    for alias in COURSE_ALIASES.get(key, []):
        if alias in database:
            return list(database[alias])

    # Pencocokan balik untuk variasi nama yang telah didefinisikan.
    for nama_alias, daftar_alias in COURSE_ALIASES.items():
        if key in daftar_alias and nama_alias in database:
            return list(database[nama_alias])

    return []


def jp_dari_database(nama_mk, database):
    """
    JP pada Mata Kuliah.xlsx dapat berupa total JP dosen untuk beberapa kelas.
    Karena itu, JP per kelas dihitung sebagai JP / jumlah kelas.
    Jika beberapa dosen mengampu mata kuliah yang sama, digunakan median
    JP per kelas agar satu nilai ekstrem tidak mengubah durasi.
    """
    data = cari_data_dosen_mk(nama_mk, database)
    nilai = []

    for item in data:
        jp_total = item["jp_total"]
        jumlah = item["jumlah_kelas"]

        if jp_total is not None and jumlah > 0:
            nilai.append(float(jp_total) / jumlah)

    if not nilai:
        return None

    return float(pd.Series(nilai).median())


def dosen_dari_database(nama_mk, database):
    data = cari_data_dosen_mk(nama_mk, database)
    hasil = {}

    for item in data:
        nama = item["dosen"]
        jumlah = item["jumlah_kelas"]
        hasil[nama] = hasil.get(nama, 0) + jumlah

    return hasil


# ============================================================
# 7. MEMBACA DAFTAR PENGAMPU DARI FILE KEBUTUHAN KELAS
# ============================================================

def parse_pengampu(teks_pengampu):
    hasil = {}
    s = txt(teks_pengampu)

    if not s or norm(s) in {"mpk", "nan", "-"}:
        return hasil

    for token in s.split(","):
        token = token.strip()

        if not token:
            continue

        # Bentuk: Nama Dosen (2) -> dua kelas.
        m = re.search(r"\((\d+)\)\s*\)?$", token)

        if m:
            jumlah = int(m.group(1))
            nama = token[:m.start()].strip()
            nama = re.sub(r"\s*\([^()]*$", "", nama).strip()
        else:
            # Bentuk: Antok (JAR), Haris (MAT), Rita -> satu kelas.
            jumlah = 1
            nama = re.sub(r"\s*\([^)]*\)\s*$", "", token).strip()

        if not nama or norm(nama) in {"mpk", "mat", "pmat", "jar"}:
            continue

        nama = nama_dosen_baku(nama)
        hasil[nama] = hasil.get(nama, 0) + jumlah

    return hasil


# ============================================================
# 8. MEMBACA KEBUTUHAN MATA KULIAH
# ============================================================

def baca_kebutuhan(raw):
    header = cari_header(
        raw,
        ["Nama Mata Kuliah", "Rencana kelas", "Dosen Pengampu MK"]
    )

    if header is None:
        raise ValueError("Header file kebutuhan kelas tidak ditemukan.")

    mapping = header_mapping(raw.iloc[header].tolist())

    data = []
    semester = None
    jalur = "UMUM"
    kurikulum_lama = False

    # PENTING:
    # Pada file kebutuhan kelas, penanda "SEMESTER 1" terletak
    # SEBELUM baris header tabel. Baca bagian sebelum header terlebih
    # dahulu agar mata kuliah semester 1 tidak terlewat.
    for i in range(header):
        penanda = txt(raw.iloc[i, 0])
        penanda_norm = norm(penanda)

        m_awal = re.search(r"semester\s+(\d+)", penanda, re.I)
        if m_awal:
            semester = int(m_awal.group(1))
            jalur = "UMUM"
            kurikulum_lama = False

        if "kurikulamlama" in penanda_norm:
            semester = "KL"
            jalur = "KURIKULUM_LAMA"
            kurikulum_lama = True

    for i in range(header + 1, len(raw)):
        v = raw.iloc[i].tolist()
        v += [None] * max(0, 13 - len(v))

        kolom0 = txt(v[0])
        kolom0_norm = norm(kolom0)
        nama = txt(ambil(v, mapping, "Nama Mata Kuliah", ""))

        # Penanda semester.
        m = re.search(r"semester\s+(\d+)", kolom0, re.I)
        if m:
            semester = int(m.group(1))
            jalur = "UMUM"
            kurikulum_lama = False
            continue

        # Penanda peminatan semester 5.
        if kolom0_norm.startswith("minatpengembangansistem"):
            jalur = "PS"
            continue

        if kolom0_norm.startswith("minatartificialintelligence"):
            jalur = "AI"
            continue

        if kolom0_norm.startswith("minatjaringan"):
            jalur = "JARINGAN"
            continue

        # Kurikulum lama tetap dibaca karena terdapat mata kuliah
        # Bahasa Query dan Pemrograman Analisa Data yang diminta memakai lab.
        if "kurikulumlama" in kolom0_norm:
            semester = "KL"
            jalur = "KURIKULUM_LAMA"
            kurikulum_lama = True
            continue

        # Lewati baris header berulang, total, dan baris kosong.
        if norm(nama) in {"namamatakuliah", "total", ""}:
            continue

        if semester is None or not nama:
            continue

        sks = angka(ambil(v, mapping, "SKS"), 0) or 0
        jp = angka(ambil(v, mapping, "JP"), None)
        peserta = angka(ambil(v, mapping, "Peserta semester"), 0) or 0
        shopper = angka(ambil(v, mapping, "Shopper"), 0) or 0
        total = angka(ambil(v, mapping, "Total"), None)

        if total is None or total <= 0:
            total = peserta + shopper

        # Pada kurikulum aktif gunakan Rencana kelas.
        # Pada Kurikulum Lama kolom Rencana kelas sering kosong,
        # sehingga gunakan Kebutuhan Kelas.
        rencana = angka(ambil(v, mapping, "Rencana kelas"), None)

        if (rencana is None or rencana <= 0) and kurikulum_lama:
            rencana = angka(ambil(v, mapping, "Kebutuhan Kelas"), None)

        if rencana is None or rencana <= 0:
            continue

        if total is None or total <= 0:
            raise ValueError(
                f"Jumlah mahasiswa untuk mata kuliah '{nama}' "
                f"semester {semester} tidak valid."
            )

        pengampu = txt(ambil(v, mapping, "Dosen Pengampu MK", ""))

        kode = txt(ambil(v, mapping, "Kode", ""))
        if not kode:
            kode = txt(ambil(v, mapping, "Kode lama", ""))
        if not kode:
            kode = norm(nama).upper()[:12]

        data.append({
            "semester": semester,
            "jalur": jalur,
            "kode": kode,
            "nama": nama,
            "sks": sks,
            "jp": jp,
            "total": int(math.ceil(total)),
            "rencana": int(rencana),
            "pengampu": pengampu,
            "kurikulum_lama": kurikulum_lama,
        })

    if not data:
        raise ValueError(
            "Tidak ditemukan mata kuliah dengan jumlah kelas lebih dari nol."
        )

    return data


# ============================================================
# 9. MEMBACA RUANG DAN KAPASITAS
# ============================================================

def baca_ruang(raw):
    h = cari_header(raw, ["Ruang", "Kapasitas"])

    if h is None:
        raise ValueError("Kolom Ruang dan Kapasitas tidak ditemukan.")

    headers = [norm(x) for x in raw.iloc[h].tolist()]

    i_nama = headers.index("ruang")
    i_kapasitas = headers.index("kapasitas")

    i_jenis = None
    for kemungkinan in ["jenisruang", "kategori", "tipe", "jenis"]:
        if kemungkinan in headers:
            i_jenis = headers.index(kemungkinan)
            break

    hasil = []

    for i in range(h + 1, len(raw)):
        nama = txt(raw.iloc[i, i_nama])
        kapasitas = angka(raw.iloc[i, i_kapasitas], None)

        jenis_excel = (
            txt(raw.iloc[i, i_jenis])
            if i_jenis is not None else ""
        )

        if not nama or kapasitas is None or kapasitas <= 0:
            continue

        hasil.append({
            "id": nama,
            "nama": nama,
            "kapasitas": int(kapasitas),
            "jenis": jenis_ruangan(nama, jenis_excel),
        })

    if not hasil:
        raise ValueError("Daftar ruangan kosong.")

    if len({x["id"] for x in hasil}) != len(hasil):
        raise ValueError("Ada nama ruangan yang duplikat.")

    return hasil


# ============================================================
# 10. MENYIAPKAN KELAS, DOSEN, DURASI, DAN JENIS RUANG
# ============================================================

def siapkan_data(files):
    raw = identifikasi_file(files)

    database_dosen = baca_database_dosen(raw["dosen"])
    kebutuhan = baca_kebutuhan(raw["kebutuhan"])
    ruang = baca_ruang(raw["ruang"])

    sections = []
    warnings = []

    for mk in kebutuhan:
        nama_mk = mk["nama"]

        # SUMBER UTAMA DOSEN: Mata Kuliah.xlsx.
        # Kolom dosen pada file kebutuhan kelas menjadi data pelengkap:
        # nama yang sama tidak dihitung dua kali, sedangkan dosen yang belum
        # tercatat di Mata Kuliah.xlsx tetap ditambahkan dari kolom pengampu.
        pengampu_database = dosen_dari_database(nama_mk, database_dosen)
        pengampu_kebutuhan = (
            {} if norm(mk["pengampu"]) == "mpk"
            else parse_pengampu(mk["pengampu"])
        )

        pengampu = dict(pengampu_database)
        for nama_dosen, jumlah in pengampu_kebutuhan.items():
            pengampu[nama_dosen] = max(
                pengampu.get(nama_dosen, 0),
                jumlah
            )

        # Tentukan jenis ruangan berdasarkan aturan mata kuliah.
        tipe = jenis_mata_kuliah(nama_mk)

        ruang_tipe = [
            r for r in ruang
            if ruang_diizinkan(tipe, r)
        ]

        if not ruang_tipe:
            raise ValueError(
                f"Tidak ada ruangan yang sesuai kategori '{tipe}' untuk "
                f"mata kuliah '{nama_mk}'. Periksa daftar ruangan pada Excel."
            )

        # Kapasitas kelas dihitung hanya dari jenis ruang yang boleh dipakai.
        kapasitas_maks = max(r["kapasitas"] for r in ruang_tipe)

        # Jumlah kelas tidak boleh kurang dari rencana dan kebutuhan kapasitas.
        kelas_dari_kapasitas = math.ceil(
            mk["total"] / kapasitas_maks
        )

        jumlah_dosen = sum(pengampu.values())

        n_kelas = max(
            mk["rencana"],
            kelas_dari_kapasitas,
            jumlah_dosen
        )

        # Durasi: utamakan JP dari Mata Kuliah.xlsx.
        # Karena JP di file tersebut merupakan total beban dosen untuk
        # beberapa kelas, hitung JP per kelas = JP / Jumlah Kelas.
        # Jika tidak tersedia, gunakan JP pada file kebutuhan kelas.
        jp_per_kelas = jp_dari_database(nama_mk, database_dosen)

        if jp_per_kelas is None or jp_per_kelas <= 0:
            jp_per_kelas = mk["jp"]

        if jp_per_kelas is None or jp_per_kelas <= 0:
            raise ValueError(
                f"JP untuk '{nama_mk}' tidak ditemukan pada kedua file Excel."
            )

        if DURASI_BERDASARKAN_JP:
            durasi_jam = int(math.ceil(jp_per_kelas))
        else:
            durasi_jam = int(
                math.ceil(float(jp_per_kelas) * MENIT_PER_JP / 60)
            )

        if durasi_jam < 1 or durasi_jam > SLOT:
            raise ValueError(
                f"Durasi '{nama_mk}' = {durasi_jam} jam. "
                f"Durasi harus antara 1 sampai {SLOT} jam."
            )

        # Bagi peserta secara merata ke setiap kelas.
        dasar, sisa = divmod(mk["total"], n_kelas)
        peserta_per_kelas = [
            dasar + (1 if i < sisa else 0)
            for i in range(n_kelas)
        ]

        # Susun dosen berdasarkan jumlah kelas pada data pengampu.
        daftar_dosen = []
        for nama_dosen, jumlah in pengampu.items():
            daftar_dosen.extend([nama_dosen] * jumlah)

        if not daftar_dosen:
            daftar_dosen = [None] * n_kelas

            if norm(mk["pengampu"]) == "mpk":
                info_dosen = "MPK (belum ditetapkan)"
            else:
                info_dosen = "Belum ditetapkan"

            warnings.append(
                f"{nama_mk}: dosen belum teridentifikasi; "
                "benturan dosen tidak dapat diperiksa untuk mata kuliah ini."
            )
        else:
            info_dosen = ""

            # Jika kelas lebih banyak daripada jumlah penugasan dosen,
            # dosen diulang untuk kelas tambahan. CP-SAT tetap mencegah
            # dosen yang sama mengajar pada waktu yang bersamaan.
            if len(daftar_dosen) < n_kelas:
                warnings.append(
                    f"{nama_mk}: jumlah kelas ({n_kelas}) lebih banyak "
                    f"daripada alokasi dosen ({len(daftar_dosen)}); "
                    "dosen dialokasikan bergiliran."
                )

                dasar_dosen = daftar_dosen.copy()
                while len(daftar_dosen) < n_kelas:
                    daftar_dosen.append(
                        dasar_dosen[len(daftar_dosen) % len(dasar_dosen)]
                    )

            daftar_dosen = daftar_dosen[:n_kelas]

        for i in range(n_kelas):
            kelas = label_kelas(i)
            jumlah_peserta = peserta_per_kelas[i]

            if jumlah_peserta > kapasitas_maks:
                raise ValueError(
                    f"{nama_mk} kelas {kelas}: {jumlah_peserta} mahasiswa "
                    f"melebihi kapasitas terbesar kategori {tipe} "
                    f"({kapasitas_maks})."
                )

            # Kelompok kurikulum lama dipisahkan dari semester aktif.
            if mk["semester"] == "KL":
                rombel = f"KL-{kelas}"
                semester_urut = 99
            elif mk["semester"] == 5:
                rombel = f"5-{mk['jalur']}-{kelas}"
                semester_urut = 5
            else:
                rombel = f"{mk['semester']}-{kelas}"
                semester_urut = int(mk["semester"])

            dosen = daftar_dosen[i]

            sections.append({
                "id": f"{mk['kode']}-{mk['semester']}-{mk['jalur']}-{kelas}",
                "kode": mk["kode"],
                "nama": nama_mk,
                "semester": mk["semester"],
                "semester_urut": semester_urut,
                "jalur": mk["jalur"],
                "kelas": kelas,
                "rombel": rombel,
                "dosen": dosen,
                "dosen_tampil": dosen if dosen else info_dosen,
                "peserta": jumlah_peserta,
                "durasi": durasi_jam,
                "jp": jp_per_kelas,
                "jenis_ruang": tipe,
            })

    return sections, ruang, warnings


# ============================================================
# 11. OPTIMASI JADWAL DENGAN OR-TOOLS CP-SAT
# ============================================================

def jalankan_cpsat(
    sections,
    rooms,
    bobot_sore,
    bobot_jumat,
    batas_detik,
    seed
):
    # Pemeriksaan beban mingguan sebelum membangun model.
    beban_rombel = defaultdict(int)
    beban_dosen = defaultdict(int)

    for section in sections:
        beban_rombel[section["rombel"]] += section["durasi"]

        if section["dosen"]:
            beban_dosen[section["dosen"]] += section["durasi"]

    terlalu_padat = [
        (nama, jam)
        for nama, jam in beban_rombel.items()
        if jam > SLOT * len(HARI)
    ]

    if terlalu_padat:
        raise ValueError(
            "Beban kelompok kelas melebihi waktu tersedia (50 jam/minggu): "
            + str(terlalu_padat[:10])
        )

    dosen_padat = [
        (nama, jam)
        for nama, jam in beban_dosen.items()
        if jam > SLOT * len(HARI)
    ]

    if dosen_padat:
        raise ValueError(
            "Beban dosen melebihi waktu tersedia (50 jam/minggu): "
            + str(dosen_padat[:10])
        )

    model = cp_model.CpModel()

    candidates = []
    per_section = defaultdict(list)
    resource_slots = defaultdict(list)

    for i, section in enumerate(sections):
        ruang_cocok = [
            r for r in rooms
            if ruang_diizinkan(section["jenis_ruang"], r)
            and r["kapasitas"] >= section["peserta"]
        ]

        if not ruang_cocok:
            raise ValueError(
                f"Tidak ada {section['jenis_ruang']} dengan kapasitas "
                f"minimal {section['peserta']} mahasiswa untuk "
                f"{section['nama']} kelas {section['kelas']}."
            )

        for h_idx, hari in enumerate(HARI):
            for mulai in range(SLOT - section["durasi"] + 1):
                selesai = mulai + section["durasi"]

                for ruang in ruang_cocok:
                    var = model.NewBoolVar(
                        f"x_{i}_{h_idx}_{mulai}_{len(candidates)}"
                    )

                    item = {
                        **section,
                        "var": var,
                        "hari": hari,
                        "hari_idx": h_idx,
                        "mulai": mulai,
                        "selesai": selesai,
                        "ruang": ruang["nama"],
                        "ruang_id": ruang["id"],
                        "kapasitas_ruang": ruang["kapasitas"],
                    }

                    candidates.append(item)
                    per_section[i].append(var)

                    for slot in range(mulai, selesai):
                        # HARD 1: satu ruang hanya untuk satu kelas pada satu waktu.
                        resource_slots[
                            ("RUANG", ruang["id"], hari, slot)
                        ].append(var)

                        # HARD 2: satu kelompok kelas tidak boleh bentrok.
                        resource_slots[
                            ("ROMBEL", section["rombel"], hari, slot)
                        ].append(var)

                        # HARD 3: satu dosen tidak boleh mengajar bersamaan.
                        if section["dosen"]:
                            resource_slots[
                                ("DOSEN", section["dosen"], hari, slot)
                            ].append(var)

        if not per_section[i]:
            raise ValueError(
                f"Tidak ada alternatif waktu untuk {section['nama']}."
            )

    # Setiap kelas wajib mendapat tepat satu jadwal.
    for i in range(len(sections)):
        model.AddExactlyOne(per_section[i])

    # Tidak boleh ada benturan sumber daya.
    for variables in resource_slots.values():
        if len(variables) > 1:
            model.AddAtMostOne(variables)

    # SOFT CONSTRAINT:
    # Bobot lebih tinggi = solver lebih kuat menghindari jam sore/Jumat.
    objective = []

    for c in candidates:
        jumlah_jam_sore = sum(
            1
            for s in range(c["mulai"], c["selesai"])
            if JAM_AWAL + s >= 15
        )

        penalti_sore = int(bobot_sore) * jumlah_jam_sore
        penalti_jumat = (
            int(bobot_jumat) if c["hari"] == "Jumat" else 0
        )

        penalti = penalti_sore + penalti_jumat

        if penalti:
            objective.append(penalti * c["var"])

    if objective:
        model.Minimize(sum(objective))
    else:
        model.Minimize(0)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = int(batas_detik)
    solver.parameters.num_search_workers = 8
    solver.parameters.random_seed = int(seed)

    status = solver.Solve(model)
    status_text = solver.StatusName(status)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise RuntimeError(
            f"Jadwal tidak ditemukan. Status CP-SAT: {status_text}. "
            "Periksa kapasitas lab, kapasitas ruang, durasi JP, "
            "jumlah kelas, dan beban dosen."
        )

    hasil = [
        c for c in candidates
        if solver.Value(c["var"]) == 1
    ]

    # VALIDASI ULANG SELURUH HARD CONSTRAINT.
    cek = defaultdict(list)

    for x in hasil:
        for slot in range(x["mulai"], x["selesai"]):
            cek[
                ("RUANG", x["ruang_id"], x["hari"], slot)
            ].append(x["id"])

            cek[
                ("ROMBEL", x["rombel"], x["hari"], slot)
            ].append(x["id"])

            if x["dosen"]:
                cek[
                    ("DOSEN", x["dosen"], x["hari"], slot)
                ].append(x["id"])

    bentrok = [
        (k, v)
        for k, v in cek.items()
        if len(v) > 1
    ]

    if bentrok:
        raise RuntimeError(
            f"Validasi hard constraint gagal: {bentrok[:5]}"
        )

    hasil.sort(
        key=lambda x: (
            x["hari_idx"],
            x["semester_urut"],
            x["jalur"],
            x["rombel"],
            x["mulai"]
        )
    )

    return hasil, status_text, solver.WallTime(), solver.ObjectiveValue()


# ============================================================
# 12. WARNA MATA KULIAH
# ============================================================

def warna_mata_kuliah(hasil):
    kode = sorted(set(x["kode"] for x in hasil))
    warna = {}

    for i, k in enumerate(kode):
        rgb = colorsys.hsv_to_rgb(
            (i * 0.61803398875) % 1,
            0.30,
            1.0
        )
        warna[k] = "".join(
            f"{int(v * 255):02X}" for v in rgb
        )

    return warna


# ============================================================
# 13. EKSPOR SATU FILE EXCEL JADWAL
# ============================================================

def export_excel(hasil, path, status, runtime):
    wb = Workbook()
    ws = wb.active

    ws.title = "JADWAL KULIAH"
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 75

    # 1 kolom rombel + 10 slot waktu + 1 label batas waktu 17.00.
    last_col = SLOT + 2

    ws.merge_cells(
        start_row=1,
        start_column=1,
        end_row=1,
        end_column=last_col
    )

    ws["A1"] = "JADWAL KULIAH PRODI INFORMATIKA"
    ws["A1"].font = Font(
        name="Arial",
        size=16,
        bold=True,
        color="FFFFFF"
    )
    ws["A1"].fill = PatternFill("solid", fgColor="1F4E78")
    ws["A1"].alignment = Alignment(
        horizontal="center",
        vertical="center"
    )
    ws.row_dimensions[1].height = 32

    ws.merge_cells(
        start_row=2,
        start_column=1,
        end_row=2,
        end_column=last_col
    )

    ws["A2"] = (
        f"Senin-Jumat | 07.00-17.00 | Interval 1 jam | "
        f"CP-SAT {status} | {runtime:.2f} detik"
    )
    ws["A2"].font = Font(italic=True, size=9)
    ws["A2"].alignment = Alignment(horizontal="center")

    ws.column_dimensions["A"].width = 18

    for col in range(2, last_col + 1):
        ws.column_dimensions[get_column_letter(col)].width = 13

    colors = warna_mata_kuliah(hasil)

    thin = Side(style="thin", color="808080")
    border = Border(
        left=thin,
        right=thin,
        top=thin,
        bottom=thin
    )

    header_fill = PatternFill("solid", fgColor="9FC5E8")

    def sort_rombel(r):
        bagian = r.split("-")
        if bagian[0] == "KL":
            return (99, "KL", bagian[-1])
        try:
            sem = int(bagian[0])
        except Exception:
            sem = 99
        return (sem, "-".join(bagian[1:-1]), bagian[-1])

    rombels = sorted(
        set(x["rombel"] for x in hasil),
        key=sort_rombel
    )

    row = 4

    for hari in HARI:
        ws.merge_cells(
            start_row=row,
            start_column=1,
            end_row=row,
            end_column=last_col
        )

        c = ws.cell(row, 1, hari.upper())
        c.fill = PatternFill("solid", fgColor="1F4E78")
        c.font = Font(bold=True, color="FFFFFF", size=11)
        c.alignment = Alignment(horizontal="center")
        c.border = border
        row += 1

        ws.cell(row, 1, "SEMESTER-KELAS")

        for s in range(SLOT + 1):
            ws.cell(
                row,
                s + 2,
                f"{JAM_AWAL + s:02d}:00"
            )

        for c in ws[row]:
            c.fill = header_fill
            c.font = Font(bold=True, size=8)
            c.alignment = Alignment(
                horizontal="center",
                vertical="center"
            )
            c.border = border

        row += 1

        for rombel in rombels:
            ws.cell(row, 1, rombel)
            ws.cell(row, 1).font = Font(bold=True)
            ws.cell(row, 1).alignment = Alignment(
                horizontal="center",
                vertical="center"
            )
            ws.cell(row, 1).border = border

            for col in range(2, last_col + 1):
                ws.cell(row, col).border = border

            daftar = [
                x for x in hasil
                if x["hari"] == hari and x["rombel"] == rombel
            ]

            for x in daftar:
                # Kolom B = jam 07.00, C = 08.00, dst.
                c1 = x["mulai"] + 2
                c2 = x["selesai"] + 1

                if c2 > last_col:
                    raise ValueError(
                        f"Posisi waktu {x['nama']} melewati batas kolom Excel."
                    )

                if c2 > c1:
                    ws.merge_cells(
                        start_row=row,
                        start_column=c1,
                        end_row=row,
                        end_column=c2
                    )

                cell = ws.cell(row, c1)
                cell.value = (
                    f"{x['nama']} ({x['kelas']})\n"
                    f"{x['dosen_tampil']}\n"
                    f"{x['ruang']}"
                )
                cell.fill = PatternFill(
                    "solid",
                    fgColor=colors[x["kode"]]
                )
                cell.font = Font(
                    name="Arial",
                    size=8,
                    bold=True
                )
                cell.alignment = Alignment(
                    horizontal="center",
                    vertical="center",
                    wrap_text=True
                )
                cell.border = border

                # Beri warna dan border pada semua sel yang tergabung.
                for col in range(c1, c2 + 1):
                    ws.cell(row, col).fill = PatternFill(
                        "solid",
                        fgColor=colors[x["kode"]]
                    )
                    ws.cell(row, col).border = border

            ws.row_dimensions[row].height = 48
            row += 1

        row += 1

    ws.freeze_panes = "B5"
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.orientation = "landscape"
    ws.page_setup.paperSize = ws.PAPERSIZE_A3
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 0

    wb.save(path)


# ============================================================
# 14. FUNGSI GENERATE UNTUK GRADIO
# ============================================================

def generate(files, bobot_sore, bobot_jumat, waktu, seed):
    try:
        sections, rooms, warnings = siapkan_data(files)

        hasil, status, runtime, objective = jalankan_cpsat(
            sections,
            rooms,
            bobot_sore,
            bobot_jumat,
            waktu,
            seed
        )

        export_excel(
            hasil,
            OUTPUT,
            status,
            runtime
        )

        jumlah_tanpa_dosen = sum(
            1 for x in sections if not x["dosen"]
        )

        jumlah_lab = sum(
            1 for x in sections if x["jenis_ruang"] == "LAB"
        )

        jumlah_lab_jaringan = sum(
            1 for x in sections if x["jenis_ruang"] == "LAB_JARINGAN"
        )

        jumlah_kelas_biasa = sum(
            1 for x in sections if x["jenis_ruang"] == "KELAS"
        )

        durasi_proyek = sorted(set(
            x["durasi"] for x in sections
            if norm(x["nama"]) == norm("Proyek Informatika")
        ))

        pesan = (
            "JADWAL BERHASIL DIBUAT\n\n"
            f"Status CP-SAT: {status}\n"
            f"Jumlah kelas terjadwal: {len(hasil)}\n"
            f"Ruang laboratorium biasa: {jumlah_lab} kelas\n"
            f"Ruang laboratorium jaringan: {jumlah_lab_jaringan} kelas\n"
            f"Ruang kelas biasa: {jumlah_kelas_biasa} kelas\n"
            f"Durasi Proyek Informatika: {durasi_proyek if durasi_proyek else 'tidak ada pada input'} jam\n"
            f"Kelas tanpa dosen teridentifikasi: {jumlah_tanpa_dosen}\n"
            "Validasi hard constraint: LULUS\n"
            f"Waktu solver: {runtime:.2f} detik\n"
            "File output: hasil_jadwal.xlsx\n\n"
            "Aturan durasi: JP dari Mata Kuliah.xlsx dibagi Jumlah Kelas "
            "untuk memperoleh JP per kelas, dikalikan 50 menit, lalu "
            "dibulatkan ke atas menjadi slot 1 jam. Jika tidak tersedia, "
            "program menggunakan JP pada Excel kebutuhan kelas."
        )

        if warnings:
            pesan += (
                "\n\nPERINGATAN:\n- "
                + "\n- ".join(warnings[:20])
            )

        pesan += (
            "\n\nKategori ruang:\n"
            "- LAB: Proyek Informatika, Alpro, Pembelajaran Mesin, "
            "Administrasi Basis Data, Pemrograman Analisis Data, "
            "Komputer Pelihat, NLP, Pemrosesan Citra, Generatif AI, "
            "Manajemen Data, Bahasa Query, dan mata kuliah lain dalam daftar.\n"
            "- LAB JARINGAN: Virtualisasi dan Cloud, Pemrograman Berbasis Web, "
            "Internet dan Aplikasinya, Protokol Internet, "
            "Jaringan Toleransi Tunda, Pengukuran dan Analisis Unjuk Kerja Jaringan.\n"
            "- KELAS: mata kuliah lainnya."
        )

        return OUTPUT, pesan

    except Exception as e:
        return None, f"GAGAL MEMBUAT JADWAL:\n{str(e)}"


# ============================================================
# 15. GUI GRADIO
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Academic Scheduler CP-SAT"
) as app:

    gr.Markdown("""
    # ACADEMIC SCHEDULER
    ### Automatic University Timetable Generator — CP-SAT

    Upload tiga file Excel sekaligus. Program membaca data dosen dari
    **Mata Kuliah.xlsx**, data kelas dari Excel kebutuhan kelas, dan
    kapasitas ruang dari Excel ruangan.

    **Aturan ruangan otomatis:**
    - **Lab Basis Data:** Sistem Tertanam, Pemerolehan Informasi,
      Administrasi Basis Data.
    - **Lab biasa / LKD:** Algoritma dan Pemrograman Dasar, Proyek Informatika,
      Alpro, Pemrograman Web, Pembelajaran Mesin, Pemrograman Analisis Data,
      Komputer Pelihat, NLP, Pemrosesan Citra, Generatif AI, Manajemen Data,
      Bahasa Query.
    - **Lab mana saja:** Pemrograman Perangkat Bergerak dan IoT.
    - **Lab Jaringan:** Virtualisasi dan Cloud, Pemrograman Berbasis Web,
      Internet dan Aplikasinya, Protokol Internet, Protokol TCP/IP Suite,
      Jaringan Toleransi Tunda, Pengukuran dan Analisis Unjuk Kerja Jaringan.
    - Mata kuliah lainnya menggunakan ruang kelas biasa.

    **Durasi:** JP dari `Mata Kuliah.xlsx` dibagi `Jumlah Kelas` untuk
    memperoleh JP per kelas, lalu dikonversi dengan 1 JP = 50 menit dan
    dibulatkan ke atas menjadi slot 1 jam. Contoh Proyek Informatika:
    8 JP / 2 kelas = 4 JP per kelas = 200 menit, sehingga dijadwalkan 4 jam.
    Jika data JP dosen tidak tersedia, program memakai JP pada Excel kebutuhan kelas.
    """)

    upload = gr.File(
        label="UPLOAD 3 FILE EXCEL",
        file_types=[".xlsx"],
        file_count="multiple",
        type="filepath"
    )

    gr.Markdown("""
    **File yang harus diunggah:**
    1. `Mata Kuliah.xlsx`
    2. `kebutuan kapasitas kelas tiap mata kuliah.xlsx`
    3. `kapaitas ruang.xlsx`
    """)

    gr.Markdown("### PENGATURAN SOFT CONSTRAINT")

    with gr.Row():
        bobot_sore = gr.Slider(
            0, 20, value=5, step=1,
            label="Bobot Menghindari Jam Sore"
        )

        bobot_jumat = gr.Slider(
            0, 20, value=2, step=1,
            label="Bobot Menghindari Hari Jumat"
        )

    with gr.Row():
        waktu = gr.Slider(
            30, 600, value=300, step=30,
            label="Batas Waktu Solver (detik)"
        )

        seed = gr.Slider(
            1, 100, value=1, step=1,
            label="Random Seed / Alternatif Jadwal"
        )

    gr.Markdown("""
    ### HARD CONSTRAINT
    1. Dosen tidak boleh mengajar bersamaan.
    2. Kelompok kelas/rombel tidak boleh memiliki jadwal bentrok.
    3. Ruangan tidak boleh digunakan dua kelas bersamaan.
    4. Kapasitas ruangan harus mencukupi jumlah mahasiswa.
    5. Durasi JP mata kuliah harus terpenuhi.
    6. Mata kuliah praktikum hanya boleh memakai laboratorium sesuai kategorinya.
    7. Sistem Tertanam, Pemerolehan Informasi, dan Administrasi Basis Data
       hanya boleh memakai Lab Basis Data.
    8. Mata kuliah jaringan hanya boleh memakai Lab Jaringan.
    9. Mata kuliah teori menggunakan ruang kelas biasa.
    """)

    tombol = gr.Button(
        "GENERATE JADWAL",
        variant="primary",
        size="lg"
    )

    status = gr.Textbox(
        label="STATUS PENJADWALAN",
        lines=18,
        interactive=False
    )

    file_hasil = gr.File(
        label="DOWNLOAD HASIL JADWAL"
    )

    tombol.click(
        fn=generate,
        inputs=[
            upload,
            bobot_sore,
            bobot_jumat,
            waktu,
            seed
        ],
        outputs=[
            file_hasil,
            status
        ]
    )


# ============================================================
# 16. JALANKAN APLIKASI
# ============================================================

app.launch(
    share=True,
    debug=False
)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
/tmp/ipykernel_2603/299905341.py:1402: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57fa186767950ea89f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
